# Paper Figures


This notebook contains the code required to make the figures in 

**Understanding Machine Learning Models Trained on DNA-Encoded Libraries for Virtual Screening** <br>
*Artur R. Menzeleev, Sathya R. Chitturi, Geraint H. M. Davies, Tony Schroeder, and Alpha A. Lee
PostEra One Broadway, 14th Floor, Cambridge, MA 02142*

## Setup and Data Loading
This portion of the notebook contains imports, code to load the data, pre-processing code

In [ ]:
%load_ext jupyter_black

In [ ]:
# imports
import pandas as pd
import seaborn as sns
import matplotlib as mpl
from matplotlib import rcParams
import numpy as np
from matplotlib import pyplot as plt

# load in the data
import os
import json
from pathlib import Path

In [ ]:
# Specify the path to your matplotlibrc.txt file
mpl.rcParams.update(mpl.rcParamsDefault)
style_file_path = "./styles/matplotlibrc_fonts.txt"
mpl.rc_file(style_file_path)
rcParams["font.family"] = "sans-serif"

# Set larger font sizes globally
plt.rcParams.update(
    {
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "figure.titlesize": 16,
    }
)

sns.set(style="whitegrid")


PLOT_CAPSIZE = 8
PLOT_MARKERSIZE = 8

In [ ]:
# UTILITY FUNCTIONS
lib_sizes_mk14 = {1: 1894242, 2: 6470656, 3: 15334340}
lib_sizes_seh = {1: 1625292, 2: 6775384, 3: 17461800}
noise_to_yield = {1: 100, 2: 82, 3: 64, 4: 60, 5: 50}


def convert_to_noise(noise_tex):
    d = {"1_1": 5, "3_2": 4, "9_2": 2, "unity": 1}
    return d.get(noise_tex, "INVALID")


def process_df(df):
    df["reads"] = df["str_reads"].apply(lambda x: int(x.split("_")[1][:-1]))
    df["noise"] = df["str_lib"].apply(
        lambda x: convert_to_noise("_".join(x.split("_")[2:]))
    )
    df["lib"] = df["str_lib"].apply(lambda x: x.split("_")[1])
    df["cycles"] = 3
    df = df[df["prediction_type"] == "screening"]
    return df


def format_ticks(reads, lib_size):
    actual_reads = reads * 1000  # Convert from 'k' notation to actual count
    sampling_percentage = (actual_reads / lib_size) * 100
    return f"$\\mathbf{{{int(reads)}k}}$\n({sampling_percentage:.1f}%)"

## Data loading

In [ ]:
# construct the data frame
# alter the experiment path as needed to load various results into records
# NOTE: this cell and the processing utility functions rely on the specific structre of the data prepared for the data


records = []

DEL_SIMULATOR_EXPERIMENT_PATHS = [
    "../data/experiments/paper/main/",
    "../data/experiments/paper/si/extended_nreads/",
    "../data/experiments/paper/si/maccs_fingerprints/",
]

for experiment_path in DEL_SIMULATOR_EXPERIMENT_PATHS:
    for data_path in Path(experiment_path).glob("*/*/*/*/*.json"):
        lib = data_path.parent.parent.parent.parent.name
        replica = data_path.parent.parent.parent.name
        reads = data_path.parent.name

        with open(data_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                except:
                    print(f"problem readin {data_path}: {line}")

                data["str_lib"] = lib
                data["str_replica"] = replica
                data["str_reads"] = reads
                data["disynthon_agg"] = str(data_path).endswith("agg_2.json")
                data["affinity_fp_method"] = (
                    "maccs" if "maccs" in str(data_path) else "morgan"
                )  # maccs| morgan
                records.append(data)

print(f"Loaded {len(DEL_SIMULATOR_EXPERIMENT_PATHS)} experiments")

df = pd.DataFrame(records)
df = process_df(df)

#### Figures 4, S2, S4, S5, S6, S7

This portion of the notebook contains the code to create Figures 4, S2, S4, S5, S6, S7

In [ ]:
figure_params_dict = {
    "F4": {
        "MODEL": "chemprop",
        "METRIC": "bedroc",
        "Y_AXIS_LABEL": "BEDROC",
        "Y_TICKS": [0, 0.2, 0.4, 0.6],
        "AFFINITY_FP_METHOD": "morgan",
        "YLIM": (-0.02, 0.75),
    },
    "S2": {
        "MODEL": "random_forest",
        "METRIC": "bedroc",
        "Y_AXIS_LABEL": "BEDROC",
        "Y_TICKS": [0, 0.2, 0.4, 0.6],
        "AFFINITY_FP_METHOD": "morgan",
        "YLIM": (-0.02, 0.75),
    },
    "S4": {
        "MODEL": "chemprop",
        "METRIC": "auc",
        "Y_AXIS_LABEL": "ROC AUC",
        "Y_TICKS": [0, 0.2, 0.4, 0.6, 0.8],
        "AFFINITY_FP_METHOD": "morgan",
        "YLIM": (-0.02, 1.0),
    },
    "S5": {
        "MODEL": "chemprop",
        "METRIC": "enrichment_1",
        "Y_AXIS_LABEL": "EF1%",
        "Y_TICKS": [0, 10, 20, 30, 40, 50],
        "YLIM": (-2.5, 60),
        "AFFINITY_FP_METHOD": "morgan",
    },
    "S6": {
        "MODEL": "chemprop",
        "METRIC": "enrichment_5",
        "Y_AXIS_LABEL": "EF5%",
        "Y_TICKS": [0, 2, 4, 6, 8, 10, 12, 14],
        "AFFINITY_FP_METHOD": "morgan",
        "YLIM": (-0.5, 16),
    },
    "S7": {
        "MODEL": "chemprop",
        "METRIC": "bedroc",
        "Y_AXIS_LABEL": "BEDROC",
        "Y_TICKS": [0, 0.2, 0.4, 0.6],
        "AFFINITY_FP_METHOD": "maccs",
        "YLIM": (-0.02, 0.75),
    },
}

In [ ]:
# SUBSET THE DATA FOR THE FIGURES

FIGURE = "S7"

FIGURE_PARAMS = figure_params_dict[FIGURE]

CYCLE = 3  # Only showing cycle 3

subset = df[
    df["lib"].isin(["a2", "b2"])
    & (df["model_type"] == FIGURE_PARAMS["MODEL"])
    & (df["affinity_fp_method"] == FIGURE_PARAMS["AFFINITY_FP_METHOD"])
]
datasets = [
    (0, subset[subset["lib"] == "a2"]),
    (1, subset[subset["lib"] == "b2"]),
]

# MAKE THE ACTUAL FIGURE

top_row_title = "(a) LIB-A2 (MK14)"
bottom_row_title = "(b) LIB-B2 (sEH)"
x_axis_label = r"$n_\mathrm{reads} \mathrm{ (thousands), } p_\mathrm{sampled}$"

fig, axes = plt.subplots(2, 4, figsize=(14, 10), sharex=False, sharey=True)

for row_idx, row_df in datasets:
    for col, noise in enumerate([1, 2, 4, 5]):
        ax = axes[row_idx, col]
        f2 = row_df[(row_df["noise"] == noise) & (row_df["cycles"] == CYCLE)]

        # Group data by 'reads' and 'disynthon_agg' to calculate median and quartiles
        grouped = (
            f2.groupby(["reads", "disynthon_agg"])[FIGURE_PARAMS["METRIC"]]
            .agg(
                median="median",
                q25=lambda x: np.percentile(x, 25),
                q75=lambda x: np.percentile(x, 75),
            )
            .reset_index()
        )

        # Create variables for aggregated and non-aggregated data
        no_agg_data = grouped[grouped["disynthon_agg"] == False].sort_values("reads")
        with_agg_data = grouped[grouped["disynthon_agg"] == True].sort_values("reads")

        # Plot line with error bars for non-aggregated data
        if not no_agg_data.empty:
            yerr_low = no_agg_data["median"] - no_agg_data["q25"]
            yerr_high = no_agg_data["q75"] - no_agg_data["median"]

            ax.errorbar(
                no_agg_data["reads"],
                no_agg_data["median"],
                yerr=[yerr_low, yerr_high],
                label="No Disynthon Agg",
                color="black",
                marker="o",
                markeredgecolor="black",
                linewidth=2,
                capsize=PLOT_CAPSIZE,
                markersize=PLOT_MARKERSIZE,
            )

        # Plot line with error bars for aggregated data
        if not with_agg_data.empty:
            yerr_low = with_agg_data["median"] - with_agg_data["q25"]
            yerr_high = with_agg_data["q75"] - with_agg_data["median"]

            ax.errorbar(
                with_agg_data["reads"],
                with_agg_data["median"],
                yerr=[yerr_low, yerr_high],
                label="With Disynthon Agg",
                color="black",
                marker="o",
                markerfacecolor="white",
                markeredgecolor="black",
                linestyle="dashed",
                linewidth=2,
                capsize=PLOT_CAPSIZE,
                markersize=PLOT_MARKERSIZE,
            )

        # Fill gray area between curves if both datasets exist
        if not no_agg_data.empty and not with_agg_data.empty:
            # Find common read values between the two datasets
            common_reads = sorted(
                set(no_agg_data["reads"]).intersection(set(with_agg_data["reads"]))
            )

            if common_reads:
                # Get median values for common read points
                no_agg_medians = no_agg_data[
                    no_agg_data["reads"].isin(common_reads)
                ].sort_values("reads")
                with_agg_medians = with_agg_data[
                    with_agg_data["reads"].isin(common_reads)
                ].sort_values("reads")

                # Fill between the curves
                ax.fill_between(
                    common_reads,
                    no_agg_medians["median"].values,
                    with_agg_medians["median"].values,
                    color="#a6bddb",
                    alpha=0.3,
                    label="Performance Gap",
                )

        # Titles and labels with larger fonts
        ax.set_title(
            r"$\gamma_{\mathrm{step}}$" + f" = {noise_to_yield[noise]}%",
            fontsize=14,
            style="italic",
            pad=8,
        )

        ax.tick_params(axis="both", which="major", labelsize=12)

        # Only show y-label on the first column
        if col == 0:
            ax.set_ylabel(FIGURE_PARAMS["Y_AXIS_LABEL"], fontsize=14)
        else:
            ax.set_ylabel("")

        # Remove individual x-axis labels from all subplots
        ax.set_xlabel("")

        ax.legend().remove()

# Set x-axis to log scale and ticks for all subplots with sampling percentages
for i, ax in enumerate(axes.flatten()):
    ax.set_xscale("log")
    ax.set_xticks([10, 20, 50, 100, 200, 500])
    ax.set_yticks(FIGURE_PARAMS["Y_TICKS"])

    # Determine row and appropriate library size
    row = i // 4  # 4 columns per row
    if row == 0:  # SEH dataset
        lib_size = lib_sizes_seh[2]
    else:  # MK14 dataset
        lib_size = lib_sizes_mk14[2]  # Medium size

    # Format tick labels with sampling percentage
    tick_labels = [
        format_ticks(reads, lib_size) for reads in [10, 20, 50, 100, 200, 500]
    ]
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=8)

# First apply tight_layout to get the main plot elements positioned
plt.tight_layout()

# Adjust subplots to create more space at top for row titles and bottom for legend and common x-label
plt.subplots_adjust(wspace=0.05, hspace=0.7, top=0.85, bottom=0.20)

# Now add row titles AFTER adjusting the layout with improved positioning
y_pos_top = 0.9
y_pos_bottom = 0.52

fig.text(
    0.5,
    y_pos_top,
    top_row_title,
    fontsize=16,
    fontweight="bold",
    ha="center",
)

fig.text(
    0.5,
    y_pos_bottom,
    bottom_row_title,
    fontsize=16,
    fontweight="bold",
    ha="center",
)

# Add a single, centered x-axis label for the entire figure
fig.text(
    0.5,
    0.12,
    x_axis_label,
    fontsize=16,
    ha="center",
)

# Create the legend handles manually for better control
handles = [
    plt.Line2D(
        [],
        [],
        color="black",
        marker="o",
        markeredgecolor="black",
        linestyle="-",
        linewidth=2,
        markersize=8,
        label=r"$z_{\mathrm{da}}=0$",
    ),
    plt.Line2D(
        [],
        [],
        color="black",
        marker="o",
        markerfacecolor="white",
        markeredgecolor="black",
        linestyle="dashed",
        linewidth=2,
        markersize=8,
        label=r"$z_{\mathrm{da}}=1$",
    ),
    plt.Rectangle((0, 0), 1, 1, color="gray", alpha=0.1, label="Performance Gap"),
]

# Add the legend with adjusted position and larger font size
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.05),
    ncol=3,
    frameon=False,
    fontsize=14,
)

plt.ylim(FIGURE_PARAMS["YLIM"])
plt.savefig(f"Figure{FIGURE}_202607.pdf", dpi=1000, bbox_inches="tight")

plt.show()

## Figures 6, S8
 
This portion of the notebook contains the code to create Figures 6 and S8 starting from the same pre-loaded dataset

In [ ]:
CYCLE = 3
NOISE = 2
MODEL_TYPE = "chemprop"
AFFINITY_FP_METHOD = "morgan"
METRIC = "bedroc"
y_axis_label = "BEDROC"
x_axis_label = r"$n_\mathrm{reads} \mathrm{ (thousands), } p_\mathrm{sampled}$"

# Updated library size names with correct mappings
lib_size_names_a = {
    1: "LIB-A1 (~1.9M)",
    2: "LIB-A2 (~6.5M)",
    3: "LIB-A3 (~15M)",
}
lib_size_names_b = {
    1: "LIB-B1 (~1.6M)",
    2: "LIB-B2 (~6.8M)",
    3: "LIB-B3 (~17M)",
}

subset = df[
    df["lib"].isin(["a1", "a2", "a3", "b1", "b2", "b3"])
    & (df["model_type"] == MODEL_TYPE)
    & (df["noise"] == NOISE)
    & (df["affinity_fp_method"] == AFFINITY_FP_METHOD)
]

df_a = subset[subset["lib"].isin(["a1", "a2", "a3"])]
df_b = subset[subset["lib"].isin(["b1", "b2", "b3"])]

# Plot datasets on respective rows
datasets = [
    (1, df_b),
    (0, df_a),
]

# Group data for MK14 and SEH

grouped_a_extended = (
    df_a.groupby(["reads", "disynthon_agg", "lib"])["bedroc"]
    .agg(
        median="median",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75),
    )
    .reset_index()
)

grouped_a = grouped_a_extended[
    grouped_a_extended["reads"].isin([10, 20, 50, 100, 200, 500])
]

grouped_b = (
    df_b.groupby(["reads", "disynthon_agg", "lib"])["bedroc"]
    .agg(
        median="median",
        q25=lambda x: np.percentile(x, 25),
        q75=lambda x: np.percentile(x, 75),
    )
    .reset_index()
)

grouped_a_extended["lib_size"] = grouped_a_extended["lib"].apply(lambda x: int(x[-1:]))
grouped_a["lib_size"] = grouped_a["lib"].apply(lambda x: int(x[-1:]))
grouped_b["lib_size"] = grouped_b["lib"].apply(lambda x: int(x[-1:]))

### Figure 6

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 10), sharey=True)

# Plot MK14 datasets (first row) - LIB-A
for i, lib_size in enumerate([1, 2, 3]):
    ax = axes[0, i]
    subset_no_agg = grouped_a[
        (grouped_a["disynthon_agg"] == False) & (grouped_a["lib_size"] == lib_size)
    ]
    subset_with_agg = grouped_a[
        (grouped_a["disynthon_agg"] == True) & (grouped_a["lib_size"] == lib_size)
    ]

    if not subset_no_agg.empty:
        yerr_low = subset_no_agg["median"] - subset_no_agg["q25"]
        yerr_high = subset_no_agg["q75"] - subset_no_agg["median"]
        ax.errorbar(
            subset_no_agg["reads"],
            subset_no_agg["median"],
            yerr=[yerr_low, yerr_high],
            label="No Disynthon Agg",
            color="black",
            marker="o",
            markeredgecolor="black",
            linewidth=2,
            capsize=PLOT_CAPSIZE,
            markersize=PLOT_MARKERSIZE,
        )
    if not subset_with_agg.empty:
        yerr_low = subset_with_agg["median"] - subset_with_agg["q25"]
        yerr_high = subset_with_agg["q75"] - subset_with_agg["median"]
        ax.errorbar(
            subset_with_agg["reads"],
            subset_with_agg["median"],
            yerr=[yerr_low, yerr_high],
            label="With Disynthon Agg",
            color="black",
            marker="o",
            markerfacecolor="white",
            markeredgecolor="black",
            linestyle="dashed",
            linewidth=2,
            capsize=PLOT_CAPSIZE,
            markersize=PLOT_MARKERSIZE,
        )
        # Fill gray area between curves
        if not subset_no_agg.empty:
            ax.fill_between(
                subset_no_agg["reads"],
                subset_no_agg["median"],
                subset_with_agg["median"],
                color="#a6bddb",
                alpha=0.3,
            )
    ax.set_title(lib_size_names_a[lib_size], fontsize=18)
    ax.set_xscale("log")
    tick_values = subset_no_agg["reads"].tolist() + subset_with_agg["reads"].tolist()
    tick_labels = [
        format_ticks(reads, lib_sizes_mk14[lib_size]) for reads in tick_values
    ]
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=8)
    if i == 0:
        ax.set_ylabel("BEDROC", fontsize=16)

# Plot SEH datasets (second row) - LIB-B
for i, lib_size in enumerate([1, 2, 3]):
    ax = axes[1, i]
    subset_no_agg = grouped_b[
        (grouped_b["disynthon_agg"] == False) & (grouped_b["lib_size"] == lib_size)
    ]
    subset_with_agg = grouped_b[
        (grouped_b["disynthon_agg"] == True) & (grouped_b["lib_size"] == lib_size)
    ]

    if not subset_no_agg.empty:
        yerr_low = subset_no_agg["median"] - subset_no_agg["q25"]
        yerr_high = subset_no_agg["q75"] - subset_no_agg["median"]
        ax.errorbar(
            subset_no_agg["reads"],
            subset_no_agg["median"],
            yerr=[yerr_low, yerr_high],
            label=r"$z_{\text{da}} = 0$",
            color="black",
            marker="o",
            markeredgecolor="black",
            linewidth=2,
            capsize=8,
            markersize=8,
        )
    if not subset_with_agg.empty:
        yerr_low = subset_with_agg["median"] - subset_with_agg["q25"]
        yerr_high = subset_with_agg["q75"] - subset_with_agg["median"]
        ax.errorbar(
            subset_with_agg["reads"],
            subset_with_agg["median"],
            yerr=[yerr_low, yerr_high],
            label=r"$z_{\text{da}} = 1$",
            color="black",
            marker="o",
            markerfacecolor="white",
            markeredgecolor="black",
            linestyle="dashed",
            linewidth=2,
            capsize=8,
            markersize=8,
        )
        # Fill gray area between curves
        if not subset_no_agg.empty and not subset_with_agg.empty:
            ax.fill_between(
                subset_no_agg["reads"],
                subset_no_agg["median"],
                subset_with_agg["median"],
                color="#a6bddb",
                alpha=0.4,
            )
    ax.set_title(lib_size_names_b[lib_size], fontsize=18)
    ax.set_xscale("log")
    tick_values = subset_no_agg["reads"].tolist() + subset_with_agg["reads"].tolist()
    tick_labels = [
        format_ticks(reads, lib_sizes_seh[lib_size]) for reads in tick_values
    ]
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=8)
    if i == 0:
        ax.set_ylabel("BEDROC", fontsize=16)

fig.text(
    0.5, 0.96, "(a) LIB-A Datasets (MK14)", ha="center", fontsize=20, fontweight="bold"
)
fig.text(
    0.5, 0.52, "(b) LIB-B Datasets (sEH)", ha="center", fontsize=20, fontweight="bold"
)

# Add legend
handles = [
    plt.Line2D(
        [],
        [],
        color="black",
        marker="o",
        markeredgecolor="black",
        linewidth=2,
        markersize=8,
        label=r"$z_{\text{da}} = 0$",
    ),
    plt.Line2D(
        [],
        [],
        color="black",
        marker="o",
        markerfacecolor="white",
        markeredgecolor="black",
        linestyle="dashed",
        linewidth=2,
        markersize=8,
        label=r"$z_{\text{da}} = 1$",
    ),
    plt.Rectangle((0, 0), 1, 1, color="#a6bddb", alpha=0.3, label="Performance Gap"),
]
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=3,
    fontsize=14,
    frameon=False,
)

fig.text(
    0.50,
    0.08,
    r"$n_{\text{reads}}$ (thousands), $p_{\text{sampled}}$ (%)",
    fontsize=16,
    ha="center",
)

plt.tight_layout()
plt.subplots_adjust(top=0.9, bottom=0.16, wspace=0.05, hspace=0.5)
plt.savefig("FigureF6_202607.pdf", dpi=1000, bbox_inches="tight")
plt.show()

### Figure S9

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)

# Plot MK14 datasets (first row) - LIB-A
for i, lib_size in enumerate([1, 2, 3]):
    ax = axes[i]
    subset_no_agg = grouped_a_extended[
        (grouped_a_extended["disynthon_agg"] == False)
        & (grouped_a_extended["lib_size"] == lib_size)
    ]
    subset_with_agg = grouped_a_extended[
        (grouped_a_extended["disynthon_agg"] == True)
        & (grouped_a_extended["lib_size"] == lib_size)
    ]

    if not subset_no_agg.empty:
        yerr_low = subset_no_agg["median"] - subset_no_agg["q25"]
        yerr_high = subset_no_agg["q75"] - subset_no_agg["median"]
        ax.errorbar(
            subset_no_agg["reads"],
            subset_no_agg["median"],
            yerr=[yerr_low, yerr_high],
            label="No Disynthon Agg",
            color="black",
            marker="o",
            markeredgecolor="black",
            linewidth=2,
            capsize=PLOT_CAPSIZE,
            markersize=PLOT_MARKERSIZE,
        )
    if not subset_with_agg.empty:
        yerr_low = subset_with_agg["median"] - subset_with_agg["q25"]
        yerr_high = subset_with_agg["q75"] - subset_with_agg["median"]
        ax.errorbar(
            subset_with_agg["reads"],
            subset_with_agg["median"],
            yerr=[yerr_low, yerr_high],
            label="With Disynthon Agg",
            color="black",
            marker="o",
            markerfacecolor="white",
            markeredgecolor="black",
            linestyle="dashed",
            linewidth=2,
            capsize=PLOT_CAPSIZE,
            markersize=PLOT_MARKERSIZE,
        )
        # Fill gray area between curves
        if not subset_no_agg.empty:
            ax.fill_between(
                subset_no_agg["reads"],
                subset_no_agg["median"],
                subset_with_agg["median"],
                color="#a6bddb",
                alpha=0.3,
            )
    ax.set_title(lib_size_names_a[lib_size], fontsize=18)
    ax.set_xscale("log")
    tick_values = subset_no_agg["reads"].tolist() + subset_with_agg["reads"].tolist()
    tick_labels = [
        format_ticks(reads, lib_sizes_mk14[lib_size]) for reads in tick_values
    ]
    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels, rotation=0, fontsize=6)
    if i == 0:
        ax.set_ylabel("BEDROC", fontsize=16)


fig.text(
    0.5, 0.96, "LIB-A Datasets (MK14)", ha="center", fontsize=20, fontweight="bold"
)


# Add legend
handles = [
    plt.Line2D(
        [],
        [],
        color="black",
        marker="o",
        markeredgecolor="black",
        linewidth=2,
        markersize=8,
        label=r"$z_{\text{da}} = 0$",
    ),
    plt.Line2D(
        [],
        [],
        color="black",
        marker="o",
        markerfacecolor="white",
        markeredgecolor="black",
        linestyle="dashed",
        linewidth=2,
        markersize=8,
        label=r"$z_{\text{da}} = 1$",
    ),
    plt.Rectangle((0, 0), 1, 1, color="#a6bddb", alpha=0.3, label="Performance Gap"),
]
fig.legend(
    handles=handles,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.02),
    ncol=3,
    fontsize=14,
    frameon=False,
)

fig.text(
    0.50,
    0.16,
    r"$n_{\text{reads}}$ (thousands), $p_{\text{sampled}}$ (%)",
    fontsize=16,
    ha="center",
)

plt.tight_layout()
plt.subplots_adjust(top=0.85, bottom=0.32, wspace=0.05, hspace=0.5)
# as a pdf
plt.savefig("FigureS9_202607.pdf", dpi=1000, bbox_inches="tight")
plt.show()

## Figure S3
This portion of the notebook contains the code to create Figure S3 starting from the same pre-loaded dataset

In [ ]:
subset = df[
    (
        df["lib"].isin(
            [
                "a2",
                "b2",
            ]
        )
    )
    & (df["affinity_fp_method"] == "morgan")
]


rf_subset = subset[subset["model_type"] == "random_forest"]
chemprop_subset = subset[subset["model_type"] == "chemprop"]

merged_df = pd.merge(
    rf_subset,
    chemprop_subset,
    on=["noise", "reads", "cycles", "str_replica", "disynthon_agg", "lib"],
    suffixes=("_rf", "_gnn"),
)

merged_df.shape

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=False, sharey=False)

titles = ["LIB-A2 (MK14)", "LIB-B2 (sEH)"]
for idx, lib in enumerate(["a2", "b2"]):
    plot_subset = merged_df[merged_df["lib"] == lib]
    ax = axes[idx]
    # Plot 45-degree line
    min_val = min(plot_subset["bedroc_gnn"].min(), plot_subset["bedroc_rf"].min())
    max_val = max(plot_subset["bedroc_gnn"].max(), plot_subset["bedroc_rf"].max())
    ax.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.7)
    ax.plot([min_val, max_val], [min_val, max_val], "k--", alpha=0.7)

    # Scatter plot
    ax.scatter(
        plot_subset["bedroc_rf"],
        plot_subset["bedroc_gnn"],
        alpha=0.7,
        edgecolors="black",
    )

    # Add title and labels
    ax.set_title(titles[idx], fontsize=14)
    ax.set_xlabel("RF BEDROC Score", fontsize=12)
    ax.set_ylabel("GNN BEDROC Score", fontsize=12)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.savefig("FigureS3_202607.pdf", dpi=1000, bbox_inches="tight")
plt.show()